### Skyline – Lösung Schritt für Schritt

*31. Bundeswettbewerb Informatik (2012), 1. Runde, Junioraufgabe 1*

Das Wahrzeichen von Boomtown ist die Needle, ein schlanker, 100 m hoher Turm. Nach ihrem Bau wurde einst beschlossen, dass
kein Gebäude in Boomtown höher als die Needle sein darf. Nun sollen viele neue Gebäude in Boomtown errichtet werden, und die
Bauherren wollen höher hinaus.

Der Rat von Boomtown überlegt, dass Gebäude, die weiter von der Needle weg sind, höher sein dürfen. So können die Bauherren
befriedigt werden, und gleichzeitig bleibt die Needle als markantes Wahrzeichen der Stadt erhalten. Der Rat erlässt also die
Vorschrift, dass mit jeden 100 m Abstand von der Needle 1 m höher gebaut werden darf. Also ist die maximal erlaubte Höhe bei
einem Abstand von 0 m bis 99 m: 100 m, bei einem Abstand von 100 m bis 199 m: 101 m usw.

Nun soll für die vielen Bauanfragen entschieden werden, wie hoch das Gebäude jeweils sein darf. Zum Glück hatten die
Stadtväter von Boomtown einst ein Koordinatensystem mit der Einheitslänge 1 m eingeführt, in dem die Needle die Koordinaten
(0,0) hat. Für die Grundrisse aller Gebäude gelten folgende Regeln:

- sie sind rechteckig;
- ihre Eckpunkte haben ganzzahlige Koordinaten;
- ihre Seiten liegen parallel zu den Achsen des Koordinatensystems.

Drei Beispielgebäude (angegeben durch die Eckpunkte):

    (−80, 100); (−20, 100); (−20, 160); (−80, 160)
    (60, 160); (120, 160); (120, 250); (60, 250)
    (−400, 200); (−300, 200); (−300, 300); (−400, 300)

#### Junioraufgabe 1

Schreibe ein Programm, das für eine Liste von Grundrissen die erlaubten Höhen der Gebäude berechnet.
Für das Beispiel lautet die richtige Ausgabe:

    101
    101
    103

Beispieldaten gibt es zu dieser Aufgabe nicht.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Abstand zum Ursprung:** Der Abstand eines Punktes `(x, y)` von der Needle `(0, 0)` ist nach Pythagoras `Wurzel(x² + y²)`.
- **Der nächste Punkt eines Rechtecks:** Entscheidend ist der **kleinste** Abstand der Gebäudefläche zur Needle. Das ist meist eine Ecke, aber nicht immer: Liegt das Gebäude über (oder neben) der Needle, ist der nächste Punkt
  ein Punkt auf einer Seite (zum Beispiel `(0, 190)` bei einem Gebäude von x = −100 bis 100 und y = 190 bis 300). Eine einfache Art, das richtig zu machen: In jeder Richtung (x und y) nimmt man die Koordinate, die der Null am nächsten liegt.
- **Eingabe normalisieren:** Aus den Eckpunkten bestimmt man mit `min` und `max` `xmin`, `xmax`, `ymin`, `ymax`. Dann ist die Reihenfolge der Ecken egal.
- **Abrunden ganzzahlig:** `//` teilt ohne Nachkommastellen. Mit `math.isqrt` (ganzzahlige Quadratwurzel) vermeidet man Rundungsfehler bei Kommazahlen.
- **Grenzfälle** am Rand eines Bereichs (genau 100 m, 199 m, 200 m).

**Nützliche Funktionen und Bibliotheken**
- `import math`, `math.sqrt`, `math.isqrt`, `min`, `max`, `abs`, `//`
- In `split()` zerlegt man eine Zeile in Zahlen, `int(...)` wandelt sie um.

## Die Idee

1. Aus den Eckpunkten bestimmen wir das Rechteck `[xmin, xmax] × [ymin, ymax]`.
2. Der nächste Punkt zur Needle: In x-Richtung ist es `0`, wenn `xmin ≤ 0 ≤ xmax` gilt (das Gebäude liegt „über“ der Needle), sonst die Koordinate mit dem kleineren Betrag, also `xmin` (wenn `xmin > 0`) oder `xmax` (wenn `xmax < 0`).
   Dasselbe in y-Richtung. Der nächste Punkt ist dann `(nx, ny)`.
3. Der kleinste Abstand ist `e = Wurzel(nx² + ny²)`.
4. Die erlaubte Höhe ist `100 + (e // 100)`: 100 Meter plus ein Meter für je **volle** 100 Meter Abstand (`//` rundet ab).

**Beispiel** (zweites Gebäude, Ecken (60, 160) bis (120, 250)): `xmin = 60 > 0`, also `nx = 60`. `ymin = 160 > 0`, also `ny = 160`. Der Abstand ist `Wurzel(60² + 160²) = Wurzel(29200) ≈ 170,9`. Das ist weniger als 200, also eine
volle 100 m: Höhe `100 + 1 = 101`. ✓

## Schritt 1: Ein Gebäude einlesen

Die Zeile enthält die Eckpunkte als Zahlen hintereinander, zum Beispiel `-80 100 -20 100 -20 160 -80 160` (vier Punkte mit je x und y). Die Zahlen auf den Positionen 0, 2, 4, ... sind die x-Koordinaten, die auf den Positionen
1, 3, 5, ... die y-Koordinaten. Mit `zahlen[0::2]` (jedes zweite Element ab 0) und `zahlen[1::2]` holen wir sie. Wer nur zwei gegenüberliegende Ecken angibt, kommt ebenfalls zurecht.

In [ ]:
def lies_gebaeude(zeile):
    zahlen = []
    for wort in zeile.replace(';', ' ').replace('(', ' ').replace(')', ' ').replace(',', ' ').split():
        zahlen.append(int(wort))
    xs = zahlen[0::2]
    ys = zahlen[1::2]
    return min(xs), max(xs), min(ys), max(ys)


print(lies_gebaeude('-80 100 -20 100 -20 160 -80 160'))
print(lies_gebaeude('(60, 160); (120, 250)'))             # nur zwei gegenüberliegende Ecken

(-80, -20, 100, 160)
(60, 120, 160, 250)


## Schritt 2: Die erlaubte Höhe

`naechster_wert(minimum, maximum)` liefert die Koordinate des Rechtecks, die der Null am nächsten liegt (0, wenn die Null im Bereich liegt).
Der Abstand zum Quadrat ist eine ganze Zahl: `nx * nx + ny * ny`. Daraus bekommen wir mit `math.isqrt` die ganzzahlige Quadratwurzel (auf die nächste ganze Zahl abgerundet). Die Division durch 100 mit `//` rundet wieder ab.
Das ist genau die Rechnung `floor(Wurzel(...) / 100)`, aber **ohne Kommazahlen**, deshalb gibt es keine Rundungsfehler an den Grenzen (zum Beispiel bei genau 100 m).

In [ ]:
import math


def naechster_wert(minimum, maximum):
    if minimum <= 0 <= maximum:
        return 0
    if minimum > 0:
        return minimum
    return maximum                  # beide Werte sind negativ: der größere liegt näher an 0


def erlaubte_hoehe(xmin, xmax, ymin, ymax):
    nx = naechster_wert(xmin, xmax)
    ny = naechster_wert(ymin, ymax)
    abstand_ganz = math.isqrt(nx * nx + ny * ny)       # Wurzel, abgerundet auf eine ganze Zahl
    return 100 + abstand_ganz // 100


for zeile in ['-80 100 -20 100 -20 160 -80 160',
              '60 160 120 160 120 250 60 250',
              '-400 200 -300 200 -300 300 -400 300']:
    print(erlaubte_hoehe(*lies_gebaeude(zeile)))

101
101
103


Das ist die richtige Ausgabe aus der Aufgabenstellung: 101, 101 und 103.

## Schritt 3: Die Liste verarbeiten

`skyline(text)` geht alle Zeilen des Textes durch (`text.splitlines()`), überspringt leere Zeilen und gibt für jedes Gebäude die erlaubte Höhe aus. Die Eingabe kann auch aus einer Datei kommen (`f.read()`).

In [ ]:
def skyline(text):
    hoehen = []
    for zeile in text.splitlines():
        if zeile.strip() != '':
            hoehen.append(erlaubte_hoehe(*lies_gebaeude(zeile)))
    return hoehen


beispiel = """(-80, 100); (-20, 100); (-20, 160); (-80, 160)
(60, 160); (120, 160); (120, 250); (60, 250)
(-400, 200); (-300, 200); (-300, 300); (-400, 300)"""

for hoehe in skyline(beispiel):
    print(hoehe)

101
101
103


## Schritt 4: Eigene Tests

Die Lösungshinweise verlangen mindestens zwei eigene Beispiele, darunter eines, das auf einer Achse liegt (also in zwei Quadranten). Wir testen besonders die Stellen, an denen Fehler leicht passieren:

| Gebäude | warum interessant | erwartet |
|---|---|---|
| (−100, 195) bis (100, 300) | liegt über der Needle (zwei Quadranten), nächster Punkt `(0, 195)`, Abstand 195 | 101 |
| (195, −100) bis (300, 100) | dasselbe auf der anderen Achse | 101 |
| (100, 0) bis (200, 50) | genau 100 m entfernt | 101 |
| (99, 0) bis (150, 50) | knapp unter 100 m | 100 |
| (−199, 0) bis (−150, 10) | Abstand 150 | 101 |
| (−200, −200) bis (−1, −1) | Ecke (−1, −1) ist am nächsten, Abstand 1,4 | 100 |
| (0, 500) bis (50, 600) | an der y-Achse, Abstand 500 | 105 |
| (−1000, −1000) bis (−900, −900) | alles negativ, nächster Punkt (−900, −900) | 112 |

In [ ]:
tests = [('-100 195 100 300', 101), ('195 -100 300 100', 101), ('100 0 200 50', 101), ('99 0 150 50', 100),
         ('-199 0 -150 10', 101), ('-200 -200 -1 -1', 100), ('0 500 50 600', 105), ('-1000 -1000 -900 -900', 112)]

for zeile, erwartet in tests:
    hoehe = erlaubte_hoehe(*lies_gebaeude(zeile))
    print(f'{zeile:24} -> {hoehe}  (erwartet {erwartet})', 'ok' if hoehe == erwartet else 'FEHLER')

-100 195 100 300         -> 101  (erwartet 101) ok
195 -100 300 100         -> 101  (erwartet 101) ok
100 0 200 50             -> 101  (erwartet 101) ok
99 0 150 50              -> 100  (erwartet 100) ok
-199 0 -150 10           -> 101  (erwartet 101) ok
-200 -200 -1 -1          -> 100  (erwartet 100) ok
0 500 50 600             -> 105  (erwartet 105) ok
-1000 -1000 -900 -900    -> 112  (erwartet 112) ok


Alle Tests stimmen. Ein Gebäude, das den Nullpunkt selbst enthält (zum Beispiel von (−10, −10) bis (10, 10)), ergäbe die Höhe 100 (Abstand 0). Der Rat von Boomtown würde das nie erlauben, denn die Needle müsste weichen.
Das Programm muss diesen Fall nicht besonders behandeln.

## Für Neugierige: Eine zweite Rechnung zur Kontrolle

Die Lösungshinweise rechnen anders: Sie berechnen die Abstände der **vier Ecken** und prüfen zusätzlich, ob eine Seite des Rechtecks eine Achse schneidet (dann ist der Schnittpunkt mit der Achse näher).
Wir bauen diese Rechnung nach (mit Kommazahlen und `math.sqrt`) und vergleichen beide Verfahren an vielen Zufallsgebäuden.

In [ ]:
import random


def erlaubte_hoehe_ecken(xmin, xmax, ymin, ymax):
    kleinster = None
    ecken = [(xmin, ymin), (xmax, ymin), (xmax, ymax), (xmin, ymax)]
    kandidaten = list(ecken)
    # Schnittpunkte der Seiten mit den Achsen
    if xmin < 0 < xmax:
        kandidaten.append((0, ymin))
        kandidaten.append((0, ymax))
    if ymin < 0 < ymax:
        kandidaten.append((xmin, 0))
        kandidaten.append((xmax, 0))
    for x, y in kandidaten:
        abstand = math.sqrt(x * x + y * y)
        if kleinster is None or abstand < kleinster:
            kleinster = abstand
    return 100 + int(kleinster // 100)


random.seed(5)
gleich = True
for versuch in range(5000):
    x1 = random.randint(-1500, 1500)
    x2 = random.randint(-1500, 1500)
    y1 = random.randint(-1500, 1500)
    y2 = random.randint(-1500, 1500)
    xmin, xmax = min(x1, x2), max(x1, x2)
    ymin, ymax = min(y1, y2), max(y1, y2)
    if xmin <= 0 <= xmax and ymin <= 0 <= ymax:
        continue                                   # enthält die Needle: kommt nicht vor
    if erlaubte_hoehe(xmin, xmax, ymin, ymax) != erlaubte_hoehe_ecken(xmin, xmax, ymin, ymax):
        gleich = False
print('Beide Rechnungen liefern immer dieselbe Höhe:', gleich)

Beide Rechnungen liefern immer dieselbe Höhe: True
